# Cricket Bowling Style Classifier + Pro Matcher (Colab, Keras/GPU, cell-based)

Everything runs as normal notebook cells — functions are defined in a cell and called in
the next one. No `%%writefile` modules, no `!python script.py` subprocess calls. Train,
test, and iterate cell-by-cell like a normal notebook.

Trains on `raiyansayeed/cricket-bowling-video-dataset` for `arm` (left/right) and
`pace_type` (fast/spin). `sureshmaheshwari021/cricket-dataset` (`Bowling_action_*` vs
`batting_stance_*`) trains a bowling-action pre-check filter that gates the pipeline.

**All three trainable models (action filter, arm, pace) are Keras neural nets, trained on
GPU only — no sklearn ensembles, no CPU training path.** The notebook asserts a GPU is
attached and stops if it isn't.

The "closest pro match" is a DTW nearest-neighbor lookup — no trainable model, unaffected
by the Keras change.

**Before running: Runtime → Change runtime type → GPU (T4 or better).**

**Last section exports everything your frontend/backend needs (`analyze_video.py` +
trained model files) into a zip, generated automatically from the code you just ran in
this notebook — not a separately maintained copy.**


## 1. Install dependencies

In [1]:
!pip install -q kaggle mediapipe opencv-python-headless scikit-learn fastdtw scipy joblib tensorflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 1.3 MB/s eta 0:00:00


## 2. Verify GPU is attached

Every model in this notebook trains on GPU only. If this errors, go to
**Runtime → Change runtime type → Hardware accelerator → GPU**, then
**Runtime → Restart session**, and re-run from the top.


In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("GPUs visible to TensorFlow:", gpus)

if not gpus:
    raise RuntimeError(
        "No GPU visible to TensorFlow. Runtime -> Change runtime type -> "
        "Hardware accelerator -> GPU, then Runtime -> Restart session, and "
        "re-run all cells. This notebook has no CPU training path."
    )

for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

print("GPU OK, memory growth enabled.")


GPUs visible to TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU OK, memory growth enabled.


## 3. Upload kaggle.json and download both datasets

In [3]:
from google.colab import files
print("Upload kaggle.json")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import os
os.makedirs("data/raw/cricket-dataset", exist_ok=True)
os.makedirs("data/raw/cricket-bowling-video-dataset", exist_ok=True)

!kaggle datasets download -d sureshmaheshwari021/cricket-dataset -p data/raw/cricket-dataset --unzip
!kaggle datasets download -d raiyansayeed/cricket-bowling-video-dataset -p data/raw/cricket-bowling-video-dataset --unzip


Upload kaggle.json


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sureshmaheshwari021/cricket-dataset
License(s): MIT
100% 2.28G/2.28G [01:38<00:00, 24.9MB/s]

Dataset URL: https://www.kaggle.com/datasets/raiyansayeed/cricket-bowling-video-dataset
License(s): MIT
100% 631M/631M [00:27<00:00, 24.3MB/s]



## 4. Confirm the unzipped folder structure

Both datasets sometimes unzip with one extra nested folder. Check these paths match what
the config cell below expects — if not, fix `IMAGE_DATASET_DIR` / `VIDEO_DATASET_DIR` there.


In [4]:
for root, dirs, files_ in os.walk("data/raw"):
    depth = root.replace("data/raw", "").count(os.sep)
    if depth < 2:
        print(root, "->", dirs if dirs else f"{len(files_)} files")


data/raw -> ['cricket-bowling-video-dataset', 'cricket-dataset']
data/raw/cricket-bowling-video-dataset -> ['Shadab Khan', 'Ravichandran Ashwin', 'Mohammed Siraj', 'Wahab Riaz', 'Lockie Furguson', 'Sakib Al Hasan', 'Mehidy Hasan Miraz', 'Keshav Maharaj', 'James Pattinson', 'James Anderson', 'Dale Steyn', 'Wanindu Hasaranga', 'Mohammed Shami ', 'Adil Rasheed', 'Hasan Mahmud', 'Shamar Joseph', 'Lasith Malinga', 'Haris Rauf', 'Jofra Archer', 'Mujeeb Ur Rahman', 'Chris Woakes', 'Pat Cummins', 'Taijul Islam', 'Mark Wood', 'Naseem Shah', 'Stuart Broad', 'Brett Lee', 'Taskin Ahmed', 'Mohammad Nabi', 'Ish Sodhi', 'Trent Boult', 'Moeen Ali', 'Ravindra Jadeja', 'Jasprit Bumrah', 'Mitchell Starc', 'Kagiso Rabada', 'Nathan Ellis', 'Josh Hazlewood', 'Morne Morkel', 'James Faulkner', 'Mitchell Johnson', 'Rashid Khan', 'Mustafizur Rahman', 'Shane Watson', 'Mashrafe Bin Mortaza', 'Sam Curran', 'Nahid Rana', 'Tim Saudi', 'Shaheen Afridi', 'DJ Bravo']
data/raw/cricket-dataset -> ['Bowling_action_3', 'Bo

## 5. Config

Plain variables, no file written — every later cell just uses these names directly.


In [5]:
import os

DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")

# Dataset 1 (sureshmaheshwari021/cricket-dataset) -- images. Bowling_action_* vs
# batting_stance_* folders train the bowling-action pre-check filter.
IMAGE_DATASET_DIR = os.path.join(RAW_DIR, "cricket-dataset")

# Dataset 2 (raiyansayeed/cricket-bowling-video-dataset) -- videos, folder per real
# player, 6 deliveries each. Only labeled data the arm/pace classifiers train on.
VIDEO_DATASET_DIR = os.path.join(RAW_DIR, "cricket-bowling-video-dataset")

MODELS_DIR = "models"
CACHE_DIR = os.path.join(DATA_DIR, "cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Keras models (architecture + weights, self-contained .keras files)
ACTION_FILTER_MODEL_PATH = os.path.join(MODELS_DIR, "action_filter.keras")
ARM_MODEL_PATH = os.path.join(MODELS_DIR, "arm_classifier.keras")
PACE_MODEL_PATH = os.path.join(MODELS_DIR, "pace_classifier.keras")

# Small sidecars (StandardScaler + label order) -- Keras doesn't serialize sklearn objects
ACTION_FILTER_META_PATH = os.path.join(MODELS_DIR, "action_filter_meta.joblib")
ARM_META_PATH = os.path.join(MODELS_DIR, "arm_meta.joblib")
PACE_META_PATH = os.path.join(MODELS_DIR, "pace_meta.joblib")

REFERENCE_LIBRARY_PATH = os.path.join(MODELS_DIR, "reference_library.npz")
FEATURE_CACHE_PATH = os.path.join(CACHE_DIR, "angle_features.npz")

SEQUENCE_LENGTH = 30  # frames sampled per clip, evenly spaced

# MediaPipe Pose landmark indices actually used
LM = {
    "nose": 0,
    "l_shoulder": 11, "r_shoulder": 12,
    "l_elbow": 13, "r_elbow": 14,
    "l_wrist": 15, "r_wrist": 16,
    "l_hip": 23, "r_hip": 24,
    "l_knee": 25, "r_knee": 26,
    "l_ankle": 27, "r_ankle": 28,
    "l_heel": 29, "r_heel": 30,
    "l_foot_index": 31, "r_foot_index": 32,
}

# Fixed order -- every feature vector below relies on this order, don't reorder
# without retraining everything.
ANGLE_NAMES = [
    "l_elbow_angle", "r_elbow_angle",
    "l_knee_angle", "r_knee_angle",
    "l_shoulder_angle", "r_shoulder_angle",
    "l_hip_angle", "r_hip_angle",
    "trunk_lean_angle",
]

ARM_CLASSES = ["left", "right"]
PACE_CLASSES = ["fast", "spin"]
ACTION_FILTER_CLASSES = ["not_bowling", "bowling"]

# folder name (exact, as seen on Kaggle) -> (arm, pace_type)
# Style ground truth is public cricketing knowledge, not learned from data.
PLAYER_STYLE = {
    "Adil Rasheed":         ("right", "spin"),
    "Brett Lee":            ("right", "fast"),
    "Chris Woakes":         ("right", "fast"),
    "DJ Bravo":             ("right", "fast"),
    "Dale Steyn":           ("right", "fast"),
    "Haris Rauf":           ("right", "fast"),
    "Hasan Mahmud":         ("right", "fast"),
    "Ish Sodhi":            ("right", "spin"),
    "James Anderson":       ("right", "fast"),
    "James Faulkner":       ("right", "fast"),
    "James Pattinson":      ("right", "fast"),
    "Jasprit Bumrah":       ("right", "fast"),
    "Jofra Archer":         ("right", "fast"),
    "Josh Hazlewood":       ("right", "fast"),
    "Kagiso Rabada":        ("right", "fast"),
    "Keshav Maharaj":       ("left",  "spin"),
    "Lasith Malinga":       ("right", "fast"),
    "Lockie Furguson":      ("right", "fast"),
    "Mark Wood":            ("right", "fast"),
    "Mashrafe Bin Mortaza": ("right", "fast"),
    "Mehidy Hasan Miraz":   ("right", "spin"),
    "Mitchell Johnson":     ("left",  "fast"),
    "Mitchell Starc":       ("left",  "fast"),
    "Moeen Ali":            ("right", "spin"),
    "Mohammad Nabi":        ("right", "spin"),
    "Mohammed Shami":       ("right", "fast"),
    "Mohammed Siraj":       ("right", "fast"),
    "Morne Morkel":         ("right", "fast"),
    "Mujeeb Ur Rahman":     ("right", "spin"),
    "Mustafizur Rahman":    ("left",  "fast"),
    "Nahid Rana":           ("right", "fast"),
    "Naseem Shah":          ("right", "fast"),
    "Nathan Ellis":         ("right", "fast"),
    "Pat Cummins":          ("right", "fast"),
    "Rashid Khan":          ("right", "spin"),
    "Ravichandran Ashwin":  ("right", "spin"),
    "Ravindra Jadeja":      ("left",  "spin"),
    "Sakib Al Hasan":       ("left",  "spin"),
    "Sam Curran":           ("left",  "fast"),
    "Shadab Khan":          ("right", "spin"),
    "Shaheen Afridi":       ("left",  "fast"),
    "Shamar Joseph":        ("right", "fast"),
    "Shane Watson":         ("right", "fast"),
    "Stuart Broad":         ("right", "fast"),
    "Taijul Islam":         ("left",  "spin"),
    "Taskin Ahmed":         ("right", "fast"),
    "Tim Saudi":            ("right", "fast"),  # dataset typo for Tim Southee
    "Trent Boult":          ("left",  "fast"),
    "Wahab Riaz":           ("left",  "fast"),
    "Wanindu Hasaranga":    ("right", "spin"),
}


## 6. Pose extraction utilities

MediaPipe pose extraction + hand-crafted joint-angle features. Two outputs, used two ways:
- angle feature vector (mean/std/min/max per joint angle across a clip) -> input for the
  Keras arm/pace classifiers.
- normalized per-frame keypoint sequence -> used for DTW distance in the pro matcher.


In [6]:
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task

In [7]:
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

POSE_MODEL_PATH = "pose_landmarker.task"

_video_options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=mp_vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.4,
    min_tracking_confidence=0.4,
)

# rest of cell (unchanged) ...


def _angle(a, b, c):
    """Angle at point b, formed by points a-b-c, in degrees. Points are (x, y)."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    cosine = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def extract_landmark_sequence(video_path, sequence_length=SEQUENCE_LENGTH):
    """Returns a list of length sequence_length; each entry is either a dict
    {landmark_name: (x, y, z, visibility)} or None if pose wasn't detected on that
    sampled frame. Coordinates are MediaPipe's normalized [0,1] image coords."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    if total_frames <= 0:
        cap.release()
        return None

    wanted = set(np.linspace(0, max(total_frames - 1, 0), sequence_length).astype(int).tolist())
    results_by_frame = {}

    with mp_vision.PoseLandmarker.create_from_options(_video_options) as landmarker:
        frame_idx = 0
        while cap.isOpened() and len(results_by_frame) < len(wanted):
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx in wanted:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                timestamp_ms = int(frame_idx * 1000 / fps)
                res = landmarker.detect_for_video(mp_image, timestamp_ms)
                if res.pose_landmarks:
                    lm = res.pose_landmarks[0]  # first detected person
                    frame_dict = {
                        name: (lm[idx].x, lm[idx].y, lm[idx].z, lm[idx].visibility)
                        for name, idx in LM.items()
                    }
                else:
                    frame_dict = None
                results_by_frame[frame_idx] = frame_dict
            frame_idx += 1
    cap.release()

    ordered = [results_by_frame.get(i) for i in sorted(wanted)]
    while len(ordered) < sequence_length:
        ordered.append(ordered[-1] if ordered else None)
    return ordered[:sequence_length]


def compute_frame_angles(frame_landmarks):
    """dict of landmark -> (x,y,z,vis) -> dict of ANGLE_NAMES -> degrees."""
    if frame_landmarks is None:
        return None
    p = {k: (v[0], v[1]) for k, v in frame_landmarks.items()}
    try:
        angles = {
            "l_elbow_angle": _angle(p["l_shoulder"], p["l_elbow"], p["l_wrist"]),
            "r_elbow_angle": _angle(p["r_shoulder"], p["r_elbow"], p["r_wrist"]),
            "l_knee_angle": _angle(p["l_hip"], p["l_knee"], p["l_ankle"]),
            "r_knee_angle": _angle(p["r_hip"], p["r_knee"], p["r_ankle"]),
            "l_shoulder_angle": _angle(p["l_elbow"], p["l_shoulder"], p["l_hip"]),
            "r_shoulder_angle": _angle(p["r_elbow"], p["r_shoulder"], p["r_hip"]),
            "l_hip_angle": _angle(p["l_shoulder"], p["l_hip"], p["l_knee"]),
            "r_hip_angle": _angle(p["r_shoulder"], p["r_hip"], p["r_knee"]),
            "trunk_lean_angle": _angle(
                p["nose"],
                ((p["l_shoulder"][0] + p["r_shoulder"][0]) / 2, (p["l_shoulder"][1] + p["r_shoulder"][1]) / 2),
                ((p["l_hip"][0] + p["r_hip"][0]) / 2, (p["l_hip"][1] + p["r_hip"][1]) / 2),
            ),
        }
    except KeyError:
        return None
    return angles


def extract_angle_feature_vector(video_path):
    """Full clip -> fixed-length feature vector: mean/std/min/max per angle.
    Returns (feature_vector, n_valid_frames) or (None, 0) if pose was never detected."""
    seq = extract_landmark_sequence(video_path)
    if seq is None:
        return None, 0

    angle_rows = []
    for frame in seq:
        angles = compute_frame_angles(frame)
        if angles is not None:
            angle_rows.append([angles[name] for name in ANGLE_NAMES])

    if len(angle_rows) < 5:
        return None, len(angle_rows)

    arr = np.array(angle_rows)
    feature_vector = np.concatenate([arr.mean(axis=0), arr.std(axis=0),
                                      arr.min(axis=0), arr.max(axis=0)])
    return feature_vector, len(angle_rows)


def feature_names():
    stats = ["mean", "std", "min", "max"]
    return [f"{name}_{stat}" for stat in stats for name in ANGLE_NAMES]


def extract_normalized_pose_sequence(video_path):
    """Full clip -> (SEQUENCE_LENGTH, n_landmarks*2) array of (x, y) coords, centered on
    hip midpoint and scaled by torso length, for DTW comparison."""
    seq = extract_landmark_sequence(video_path)
    if seq is None:
        return None

    names = list(LM.keys())
    frames = []
    last_valid = None
    for frame in seq:
        if frame is None:
            frames.append(last_valid)
            continue
        hip_mid = np.array([(frame["l_hip"][0] + frame["r_hip"][0]) / 2,
                             (frame["l_hip"][1] + frame["r_hip"][1]) / 2])
        shoulder_mid = np.array([(frame["l_shoulder"][0] + frame["r_shoulder"][0]) / 2,
                                  (frame["l_shoulder"][1] + frame["r_shoulder"][1]) / 2])
        torso_len = max(np.linalg.norm(shoulder_mid - hip_mid), 1e-6)
        coords = []
        for name in names:
            x, y = frame[name][0], frame[name][1]
            coords.extend([(x - hip_mid[0]) / torso_len, (y - hip_mid[1]) / torso_len])
        frames.append(np.array(coords))
        last_valid = frames[-1]

    if all(f is None for f in frames):
        return None
    first_valid = next(f for f in frames if f is not None)
    frames = [f if f is not None else first_valid for f in frames]
    return np.array(frames)

## 7. Dataset indexing

Indexes the video dataset (player folders of delivery clips) and attaches labels from
`PLAYER_STYLE`. Grouped split so clips from the same player never span train/val
(plain random splitting would leak — 6 clips of the same player are near-duplicates).


In [8]:
from dataclasses import dataclass
from typing import List
from sklearn.model_selection import GroupKFold


@dataclass
class ClipEntry:
    path: str
    player: str
    arm: str
    pace: str


def index_video_dataset(video_dir=VIDEO_DATASET_DIR):
    if not os.path.isdir(video_dir):
        raise FileNotFoundError(
            f"{video_dir} not found. Run the dataset-download cell first, or check "
            f"VIDEO_DATASET_DIR matches the unzipped folder name."
        )

    entries = []
    skipped_unmapped = []
    for player_folder in sorted(os.listdir(video_dir)):
        player_path = os.path.join(video_dir, player_folder)
        if not os.path.isdir(player_path):
            continue
        if player_folder not in PLAYER_STYLE:
            skipped_unmapped.append(player_folder)
            continue
        arm, pace = PLAYER_STYLE[player_folder]
        for fname in sorted(os.listdir(player_path)):
            if fname.lower().endswith((".mp4", ".avi", ".mov")):
                entries.append(ClipEntry(
                    path=os.path.join(player_path, fname),
                    player=player_folder, arm=arm, pace=pace,
                ))

    if skipped_unmapped:
        print(f"WARNING: {len(skipped_unmapped)} folder(s) not in PLAYER_STYLE, skipped: "
              f"{skipped_unmapped}")

    print(f"Indexed {len(entries)} clips across {len(set(e.player for e in entries))} players.")
    return entries


def grouped_kfold_indices(entries, n_splits=5):
    """Yields (train_idx, val_idx) folds grouped by player -- zero leakage."""
    players = [e.player for e in entries]
    gkf = GroupKFold(n_splits=n_splits)
    dummy_X = list(range(len(entries)))
    for train_idx, val_idx in gkf.split(dummy_X, groups=players):
        yield train_idx, val_idx


## 8. Sanity-check the player→style label mapping against real folder names

The Kaggle dataset has a few typo'd folder names — `PLAYER_STYLE` keys must match exactly
or that player's clips get silently skipped.


In [9]:
actual_folders = set(os.listdir(VIDEO_DATASET_DIR))
mapped_folders = set(PLAYER_STYLE.keys())

print("In dataset but NOT in PLAYER_STYLE (will be skipped):")
print(actual_folders - mapped_folders or "  none - good")
print("\nIn PLAYER_STYLE but NOT in dataset (harmless, just unused):")
print(mapped_folders - actual_folders or "  none")


In dataset but NOT in PLAYER_STYLE (will be skipped):
{'Mohammed Shami '}

In PLAYER_STYLE but NOT in dataset (harmless, just unused):
{'Mohammed Shami'}


## 9. Bowling-action pre-check filter (Keras)

Binary "is this actually a bowling action" filter, trained on `Bowling_action_*`
(positive) vs `batting_stance_*` (negative) image folders. Runs on a single frame's pose
angles. This is a gate in front of the real pipeline, not part of arm/pace/pro-match.


In [10]:
import random

POSITIVE_PREFIX = "Bowling_action"
NEGATIVE_PREFIX = "batting_stance"

_image_options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.4,
)


def extract_image_angle_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    with mp_vision.PoseLandmarker.create_from_options(_image_options) as landmarker:
        res = landmarker.detect(mp_image)
    if not res.pose_landmarks:
        return None
    lm = res.pose_landmarks[0]
    frame = {name: (lm[idx].x, lm[idx].y, lm[idx].z, lm[idx].visibility)
             for name, idx in LM.items()}
    angles = compute_frame_angles(frame)
    return None if angles is None else np.array([angles[n] for n in ANGLE_NAMES], dtype="float32")


def index_action_dataset(image_dataset_dir=IMAGE_DATASET_DIR, max_per_class=1500):
    pos_paths, neg_paths = [], []
    for folder in sorted(os.listdir(image_dataset_dir)):
        folder_path = os.path.join(image_dataset_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        images = []
        for root, _, files in os.walk(folder_path):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    images.append(os.path.join(root, f))

        if folder.startswith(POSITIVE_PREFIX):
            pos_paths += images
        elif folder.startswith(NEGATIVE_PREFIX):
            neg_paths += images

    random.seed(42)
    random.shuffle(pos_paths)
    random.shuffle(neg_paths)
    return pos_paths[:max_per_class], neg_paths[:max_per_class]


def build_action_filter_model(input_dim):
    """Small dense net -- 9 angle features in, 1 sigmoid out."""
    from tensorflow import keras
    from tensorflow.keras import layers

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="action_filter")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

In [24]:
sample_dir = os.path.join(IMAGE_DATASET_DIR, "Bowling_action_1")
print(os.listdir(sample_dir)[:20])
for folder in sorted(os.listdir(IMAGE_DATASET_DIR)):
    fp = os.path.join(IMAGE_DATASET_DIR, folder)
    print(folder, os.path.isdir(fp), len(os.listdir(fp)) if os.path.isdir(fp) else "-")

['Bowling_action']
Bowling_action_1 True 1
Bowling_action_2 True 1
Bowling_action_3 True 1
Bowling_action_4 True 1
Bowling_action_5 True 1
batting_stance_1 True 1
batting_stance_2 True 1
batting_stance_3 True 1
batting_stance_4 True 1
batting_stance_5 True 1
cricket_dataset_zip_ True 1


In [11]:
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib


def train_action_filter():
    """GPU only -- trains inside tf.device("/GPU:0"), no CPU fallback.
    Returns (model, scaler) and also saves both to disk."""
    pos_paths, neg_paths = index_action_dataset(max_per_class=1500)
    print(f"Positive (bowling): {len(pos_paths)}   Negative (batting): {len(neg_paths)}")

    X, y = [], []
    items = [(p, 1) for p in pos_paths] + [(p, 0) for p in neg_paths]
    for i, (path, label) in enumerate(items):
        feat = extract_image_angle_features(path)
        if feat is not None:
            X.append(feat)
            y.append(label)
        if (i + 1) % 500 == 0:
            print(f"  processed {i + 1}/{len(items)}")

    X = np.array(X, dtype="float32")
    y = np.array(y, dtype="float32")
    print(f"Usable images: {len(X)} (rest skipped -- pose not detected)")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)

    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")

    with tf.device("/GPU:0"):
        model = build_action_filter_model(input_dim=X_train.shape[1])
        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True)
        model.fit(
            X_train, y_train,
            validation_split=0.15,
            epochs=100,
            batch_size=32,
            class_weight={0: len(y_train) / (2 * (y_train == 0).sum()),
                          1: len(y_train) / (2 * (y_train == 1).sum())},
            callbacks=[early_stop],
            verbose=2,
        )

    print("\n=== Bowling-action filter -- held-out test set ===")
    y_pred = (model.predict(X_test, verbose=0) >= 0.5).astype(int).ravel()
    print(classification_report(y_test, y_pred, target_names=ACTION_FILTER_CLASSES))
    # Caveat: images are augmented, so near-duplicates of the same source image can land
    # on both sides of the split -- treat this as a sanity check, not a formal accuracy.

    model.save(ACTION_FILTER_MODEL_PATH)
    joblib.dump({"scaler": scaler}, ACTION_FILTER_META_PATH)
    print(f"Saved: {ACTION_FILTER_MODEL_PATH}")
    print(f"Saved: {ACTION_FILTER_META_PATH}")
    return model, scaler


Run it:

In [12]:
action_model, action_scaler = train_action_filter()


Positive (bowling): 1500   Negative (batting): 1500
  processed 500/3000
  processed 1000/3000
  processed 1500/3000
  processed 2000/3000
  processed 2500/3000
  processed 3000/3000
Usable images: 2690 (rest skipped -- pose not detected)
Epoch 1/100
58/58 - 4s - 76ms/step - accuracy: 0.5347 - loss: 0.6996 - val_accuracy: 0.6502 - val_loss: 0.6595
Epoch 2/100
58/58 - 0s - 4ms/step - accuracy: 0.6167 - loss: 0.6563 - val_accuracy: 0.6749 - val_loss: 0.6356
Epoch 3/100
58/58 - 0s - 4ms/step - accuracy: 0.6572 - loss: 0.6325 - val_accuracy: 0.6749 - val_loss: 0.6204
Epoch 4/100
58/58 - 0s - 4ms/step - accuracy: 0.6796 - loss: 0.6139 - val_accuracy: 0.6842 - val_loss: 0.6026
Epoch 5/100
58/58 - 0s - 4ms/step - accuracy: 0.6873 - loss: 0.5993 - val_accuracy: 0.6780 - val_loss: 0.5871
Epoch 6/100
58/58 - 0s - 4ms/step - accuracy: 0.6873 - loss: 0.5876 - val_accuracy: 0.7059 - val_loss: 0.5807
Epoch 7/100
58/58 - 0s - 4ms/step - accuracy: 0.7042 - loss: 0.5738 - val_accuracy: 0.7028 - val_los

## 10. Arm + pace classifiers (Keras)

Dense net on 36-d angle-summary features (mean/std/min/max of 9 joint angles per clip).
GPU only. Grouped 5-fold CV (player-level, leak-free) gives the honest accuracy estimate
before the final model trains on everything.


In [13]:
from sklearn.preprocessing import LabelEncoder
import re

def build_classifier_model(input_dim, name):
    from tensorflow import keras
    from tensorflow.keras import layers
    safe_name = re.sub(r"[^A-Za-z0-9_.\-]", "_", name)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ], name=safe_name)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_feature_matrix(entries, cache_path=FEATURE_CACHE_PATH, use_cache=True):
    """Extracts (or loads cached) angle-summary features for every clip. Pose extraction
    is the slow step -- cache so repeated training runs don't re-run MediaPipe."""
    if use_cache and os.path.isfile(cache_path):
        print(f"Loading cached features from {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["X"], list(data["paths"]), list(data["players"]), \
            list(data["arms"]), list(data["paces"])

    X, paths, players, arms, paces = [], [], [], [], []
    for i, entry in enumerate(entries):
        vec, n_valid = extract_angle_feature_vector(entry.path)
        if vec is None:
            print(f"  [{i+1}/{len(entries)}] SKIPPED (pose not detected): {entry.path}")
            continue
        X.append(vec)
        paths.append(entry.path)
        players.append(entry.player)
        arms.append(entry.arm)
        paces.append(entry.pace)
        print(f"  [{i+1}/{len(entries)}] OK ({n_valid} valid frames): {entry.player}")

    X = np.array(X)
    np.savez(cache_path, X=X, paths=paths, players=players, arms=arms, paces=paces)
    print(f"Cached features to {cache_path}")
    return X, paths, players, arms, paces


def evaluate_grouped(X, y_labels, players, classes, label_name):
    class GroupedEntry:
        def __init__(self, player):
            self.player = player
    fake_entries = [GroupedEntry(p) for p in players]

    le = LabelEncoder().fit(classes)
    y = le.transform(y_labels).astype("float32")

    all_true, all_pred = [], []
    for fold, (train_idx, val_idx) in enumerate(grouped_kfold_indices(fake_entries, n_splits=5)):
        scaler = StandardScaler().fit(X[train_idx])
        X_train = scaler.transform(X[train_idx]).astype("float32")
        X_val = scaler.transform(X[val_idx]).astype("float32")

        with tf.device("/GPU:0"):
            model = build_classifier_model(X_train.shape[1], name=f"{label_name}_fold{fold}")
            model.fit(X_train, y[train_idx], epochs=60, batch_size=16, verbose=0)
            preds = (model.predict(X_val, verbose=0) >= 0.5).astype(int).ravel()

        all_true += list(y[val_idx].astype(int))
        all_pred += list(preds)

    print(f"\n=== {label_name} classifier -- 5-fold grouped-by-player CV ===")
    print(classification_report(all_true, all_pred, target_names=classes))


def train_final_model(X, y_labels, classes, model_path, meta_path, label_name):
    le = LabelEncoder().fit(classes)
    y = le.transform(y_labels).astype("float32")

    scaler = StandardScaler().fit(X)
    X_scaled = scaler.transform(X).astype("float32")

    with tf.device("/GPU:0"):
        model = build_classifier_model(X_scaled.shape[1], name=label_name)
        model.fit(X_scaled, y, epochs=80, batch_size=16, verbose=2)

    model.save(model_path)
    joblib.dump({"scaler": scaler, "classes": classes}, meta_path)
    print(f"Saved: {model_path}")
    print(f"Saved: {meta_path}")
    return model, scaler


In [14]:
def train_classifiers():
    entries = index_video_dataset()
    print("\nExtracting pose features (slow first run, cached after)...")
    X, paths, players, arms, paces = build_feature_matrix(entries)

    if len(X) < 20:
        raise RuntimeError(f"Only {len(X)} clips had usable pose data -- too few to train on.")

    evaluate_grouped(X, arms, players, ARM_CLASSES, "arm_left_right")
    evaluate_grouped(X, paces, players, PACE_CLASSES, "pace_fast_spin")

    print("\nTraining final models on all available data...")
    arm_model, arm_scaler = train_final_model(
        X, arms, ARM_CLASSES, ARM_MODEL_PATH, ARM_META_PATH, "arm_classifier")
    pace_model, pace_scaler = train_final_model(
        X, paces, PACE_CLASSES, PACE_MODEL_PATH, PACE_META_PATH, "pace_classifier")

    print("\nThe CV report above is the honest accuracy estimate -- the final models "
          "are trained on everything, so don't re-evaluate on the training clips.")
    return arm_model, arm_scaler, pace_model, pace_scaler

Run it:

In [15]:
arm_model, arm_scaler, pace_model, pace_scaler = train_classifiers()

Indexed 295 clips across 49 players.

Extracting pose features (slow first run, cached after)...
  [1/295] OK (30 valid frames): Adil Rasheed
  [2/295] OK (30 valid frames): Adil Rasheed
  [3/295] OK (30 valid frames): Adil Rasheed
  [4/295] OK (30 valid frames): Adil Rasheed
  [5/295] OK (30 valid frames): Adil Rasheed
  [6/295] OK (30 valid frames): Adil Rasheed
  [7/295] OK (30 valid frames): Brett Lee
  [8/295] OK (30 valid frames): Brett Lee
  [9/295] OK (30 valid frames): Brett Lee
  [10/295] OK (30 valid frames): Brett Lee
  [11/295] OK (30 valid frames): Brett Lee
  [12/295] OK (30 valid frames): Brett Lee
  [13/295] OK (30 valid frames): Chris Woakes
  [14/295] OK (30 valid frames): Chris Woakes
  [15/295] OK (30 valid frames): Chris Woakes
  [16/295] OK (30 valid frames): Chris Woakes
  [17/295] OK (28 valid frames): Chris Woakes
  [18/295] OK (30 valid frames): Chris Woakes
  [19/295] OK (30 valid frames): DJ Bravo
  [20/295] OK (30 valid frames): DJ Bravo
  [21/295] OK (30 


=== arm_left_right classifier -- 5-fold grouped-by-player CV ===
              precision    recall  f1-score   support

        left       0.79      0.73      0.76        66
       right       0.92      0.94      0.93       229

    accuracy                           0.89       295
   macro avg       0.85      0.84      0.84       295
weighted avg       0.89      0.89      0.89       295


=== pace_fast_spin classifier -- 5-fold grouped-by-player CV ===
              precision    recall  f1-score   support

        fast       0.81      0.87      0.84       210
        spin       0.61      0.49      0.55        85

    accuracy                           0.76       295
   macro avg       0.71      0.68      0.69       295
weighted avg       0.75      0.76      0.75       295


Training final models on all available data...
Epoch 1/80
19/19 - 4s - 221ms/step - accuracy: 0.7593 - loss: 0.5602
Epoch 2/80
19/19 - 0s - 7ms/step - accuracy: 0.8271 - loss: 0.4503
Epoch 3/80
19/19 - 0s - 7ms/st

## 11. Build the pro-matching reference library

No training -- just precomputes normalized pose sequences for all clips so an uploaded
video can be nearest-neighbor matched by DTW distance.


In [16]:
def build_reference_library():
    entries = index_video_dataset()
    sequences, players, arms, paces, paths = [], [], [], [], []

    for i, entry in enumerate(entries):
        seq = extract_normalized_pose_sequence(entry.path)
        if seq is None:
            print(f"  [{i+1}/{len(entries)}] SKIPPED (pose not detected): {entry.path}")
            continue
        sequences.append(seq)
        players.append(entry.player)
        arms.append(entry.arm)
        paces.append(entry.pace)
        paths.append(entry.path)
        print(f"  [{i+1}/{len(entries)}] OK: {entry.player}")

    sequences = np.array(sequences)
    ref = {"sequences": sequences, "players": players, "arms": arms,
           "paces": paces, "paths": paths}
    np.savez(REFERENCE_LIBRARY_PATH, **ref)
    print(f"\nSaved reference library with {len(sequences)} clips to {REFERENCE_LIBRARY_PATH}")
    return ref


Run it:

In [17]:
ref = build_reference_library()


Indexed 295 clips across 49 players.
  [1/295] OK: Adil Rasheed
  [2/295] OK: Adil Rasheed
  [3/295] OK: Adil Rasheed
  [4/295] OK: Adil Rasheed
  [5/295] OK: Adil Rasheed
  [6/295] OK: Adil Rasheed
  [7/295] OK: Brett Lee
  [8/295] OK: Brett Lee
  [9/295] OK: Brett Lee
  [10/295] OK: Brett Lee
  [11/295] OK: Brett Lee
  [12/295] OK: Brett Lee
  [13/295] OK: Chris Woakes
  [14/295] OK: Chris Woakes
  [15/295] OK: Chris Woakes
  [16/295] OK: Chris Woakes
  [17/295] OK: Chris Woakes
  [18/295] OK: Chris Woakes
  [19/295] OK: DJ Bravo
  [20/295] OK: DJ Bravo
  [21/295] OK: DJ Bravo
  [22/295] OK: DJ Bravo
  [23/295] OK: DJ Bravo
  [24/295] OK: DJ Bravo
  [25/295] OK: Dale Steyn
  [26/295] OK: Dale Steyn
  [27/295] OK: Dale Steyn
  [28/295] OK: Dale Steyn
  [29/295] OK: Dale Steyn
  [30/295] OK: Dale Steyn
  [31/295] OK: Haris Rauf
  [32/295] OK: Haris Rauf
  [33/295] OK: Haris Rauf
  [34/295] OK: Haris Rauf
  [35/295] OK: Haris Rauf
  [36/295] OK: Haris Rauf
  [37/295] OK: Hasan Mahmud
  

## 12. Full analysis pipeline

Uses the models already trained above (in memory). `generate_report` is what a frontend
upload handler calls: pre-check filter -> arm/pace classification -> closest-pro DTW
match -> comparison metrics -> one JSON report.

`load_models()` is included for reloading from disk in a fresh session (e.g. after a
runtime restart, or in the exported backend package) without retraining.


In [25]:
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (24, 26), (25, 27), (26, 28),
    (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32),
]


def load_models():
    """Reloads all three Keras models + sidecars + the reference library from disk,
    and sets them as the globals used below -- for use after a runtime restart, or
    standalone in the exported backend package."""
    global action_model, action_scaler, arm_model, arm_scaler, pace_model, pace_scaler, ref
    from tensorflow import keras
    action_model = keras.models.load_model(ACTION_FILTER_MODEL_PATH)
    action_scaler = joblib.load(ACTION_FILTER_META_PATH)["scaler"]
    arm_meta = joblib.load(ARM_META_PATH)
    arm_model = keras.models.load_model(ARM_MODEL_PATH)
    arm_scaler = arm_meta["scaler"]
    pace_meta = joblib.load(PACE_META_PATH)
    pace_model = keras.models.load_model(PACE_MODEL_PATH)
    pace_scaler = pace_meta["scaler"]
    ref = np.load(REFERENCE_LIBRARY_PATH, allow_pickle=True)
    return action_model, action_scaler, arm_model, arm_scaler, pace_model, pace_scaler, ref


def is_bowling_action(video_path, sample_frames=5):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if total <= 0:
        return True, 0.0

    wanted = set(np.linspace(0, total - 1, sample_frames).astype(int).tolist())
    cap = cv2.VideoCapture(video_path)
    votes, frame_idx = [], 0
    tmp_frame_path = video_path + "._filter_frame.jpg"
    while cap.isOpened() and len(votes) < len(wanted):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in wanted:
            cv2.imwrite(tmp_frame_path, frame)
            feat = extract_image_angle_features(tmp_frame_path)
            if feat is not None:
                feat_scaled = action_scaler.transform([feat]).astype("float32")
                prob = float(action_model.predict(feat_scaled, verbose=0)[0][0])
                votes.append(prob)
        frame_idx += 1
    cap.release()
    if os.path.isfile(tmp_frame_path):
        os.remove(tmp_frame_path)

    if not votes:
        return True, 0.0
    avg_conf = float(np.mean(votes))
    return avg_conf >= 0.5, round(avg_conf, 3)


def classify(model, scaler, classes, feature_vector):
    scaled = scaler.transform([feature_vector]).astype("float32")
    prob = float(model.predict(scaled, verbose=0)[0][0])
    pred_idx = 1 if prob >= 0.5 else 0
    confidence = prob if pred_idx == 1 else 1.0 - prob
    return {"label": classes[pred_idx], "confidence": round(confidence, 3)}


def find_closest_match(user_seq, ref):
    best_dist, best_idx = None, None
    for i, ref_seq in enumerate(ref["sequences"]):
        dist, _ = fastdtw(user_seq, ref_seq, dist=euclidean)
        if best_dist is None or dist < best_dist:
            best_dist, best_idx = dist, i
    return best_idx, best_dist


def comparison_metrics(user_seq, ref_seq):
    """All metrics are relative to the single matched reference clip, not absolute
    biomechanical measurements."""
    _, dtw_path = fastdtw(user_seq, ref_seq, dist=euclidean)
    aligned_user = np.array([user_seq[i] for i, _ in dtw_path])
    aligned_ref = np.array([ref_seq[j] for _, j in dtw_path])

    frame_dists = np.linalg.norm(aligned_user - aligned_ref, axis=1)
    similarity_pct = round(max(0.0, 100.0 - float(frame_dists.mean()) * 20), 1)
    symmetry_score = round(float(1.0 - np.std(frame_dists) / (np.mean(frame_dists) + 1e-6)), 3)
    timing_deviation_frames = round(float(np.mean(np.abs(
        np.diff([i for i, _ in dtw_path]) - np.diff([j for _, j in dtw_path])
    ))), 2) if len(dtw_path) > 1 else 0.0

    return {
        "movement_quality_similarity_pct": similarity_pct,
        "symmetry_score": symmetry_score,
        "coordination_timing_deviation_frames": timing_deviation_frames,
        "caveats": [
            "All metrics are RELATIVE to the single matched reference clip, not "
            "absolute/clinical measurements.",
            "Assumes the uploaded video's camera angle roughly matches the "
            "reference dataset's angle.",
        ],
    }


def generate_report(video_path):
    """This is what a frontend upload handler calls: report = generate_report(video_path)"""
    passed, action_conf = is_bowling_action(video_path)
    if not passed:
        return {"error": f"Doesn't look like a bowling action (confidence: {action_conf}). "
                          f"Upload a clearer bowling clip.",
                "action_filter_confidence": action_conf}

    feature_vec, n_valid = extract_angle_feature_vector(video_path)
    if feature_vec is None:
        return {"error": f"Pose not detected reliably (only {n_valid} valid frames). "
                          f"Try a clearer, more front-on video."}

    user_seq = extract_normalized_pose_sequence(video_path)

    arm_result = classify(arm_model, arm_scaler, ARM_CLASSES, feature_vec)
    pace_result = classify(pace_model, pace_scaler, PACE_CLASSES, feature_vec)

    best_idx, best_dist = find_closest_match(user_seq, ref)
    matched_player = str(ref["players"][best_idx])
    metrics = comparison_metrics(user_seq, ref["sequences"][best_idx])

    return {
        "action_filter_confidence": action_conf,
        "arm_classification": arm_result,
        "pace_classification": pace_result,
        "closest_pro_match": {
            "player": matched_player,
            "dtw_distance": round(float(best_dist), 3),
            "note": "Lower distance = closer match. Meaningful only if your video's "
                    "camera angle matches the reference clip's angle.",
        },
        "comparison": metrics,
    }


def generate_annotated_video(video_path, output_path, panel_width=380):
    report = generate_report(video_path)
    if "error" in report:
        print(report["error"])
        return None, report

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    canvas_width = width + panel_width
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (canvas_width, height))

    arm, pace = report["arm_classification"], report["pace_classification"]
    match, comp = report["closest_pro_match"], report["comparison"]

    lines = [
        (f"{arm['label'].upper()} ARM / {pace['label'].upper()}", (255, 255, 255), 0.7, 2),
        (f"Arm conf: {arm['confidence']*100:.0f}%  Pace conf: {pace['confidence']*100:.0f}%",
         (180, 180, 180), 0.45, 1),
        ("", None, 0, 0),
        (f"Closest match: {match['player']}", (120, 255, 150), 0.55, 1),
        (f"Movement quality: {comp['movement_quality_similarity_pct']}%", (120, 210, 255), 0.5, 1),
        (f"Symmetry: {comp['symmetry_score']}", (120, 210, 255), 0.5, 1),
        ("", None, 0, 0),
        ("(approximate, single-angle reference)", (120, 120, 120), 0.4, 1),
    ]

    with mp_vision.PoseLandmarker.create_from_options(_video_options) as landmarker:
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(frame_idx * 1000 / fps)
            results = landmarker.detect_for_video(mp_image, timestamp_ms)
            if results.pose_landmarks:
                lm = results.pose_landmarks[0]
                pts = [(int(p.x * width), int(p.y * height)) for p in lm]
                for a, b in POSE_CONNECTIONS:
                    cv2.line(frame, pts[a], pts[b], (0, 255, 0), 2)
                for x, y in pts:
                    cv2.circle(frame, (x, y), 3, (0, 200, 255), -1)

            canvas = np.zeros((height, canvas_width, 3), dtype=np.uint8)
            canvas[:, :width] = frame
            panel_x0 = width
            overlay = canvas.copy()
            cv2.rectangle(overlay, (panel_x0, 0), (panel_x0 + panel_width, height), (25, 25, 25), -1)
            canvas = cv2.addWeighted(overlay, 0.88, canvas, 0.12, 0)
            cv2.line(canvas, (panel_x0, 0), (panel_x0, height), (80, 80, 80), 2)

            y = 45
            for text, color, scale, thickness in lines:
                if text == "":
                    y += 18
                    continue
                cv2.putText(canvas, text, (panel_x0 + 20, y),
                            cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)
                y += int(32 * scale) + 14

            out.write(canvas)
            frame_idx += 1

    cap.release()
    out.release()

    h264_path = output_path.replace(".mp4", "_h264.mp4")
    cmd = (
        f'ffmpeg -y -loglevel error -i "{output_path}" '
        f'-vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" '
        f'-vcodec libx264 -pix_fmt yuv420p -movflags +faststart '
        f'"{h264_path}"'
    )
    ret_code = os.system(cmd)
    if ret_code != 0 or not os.path.exists(h264_path):
        print(f"ffmpeg conversion failed (exit code {ret_code}); returning raw output instead.")
        return output_path, report

    return h264_path, report

## 13. Try it on a sample video

In [26]:
print("Upload a test bowling video to run through the full pipeline")
uploaded = files.upload()
test_video_path = list(uploaded.keys())[0]

import json
report = generate_report(test_video_path)
print(json.dumps(report, indent=2))


Upload a test bowling video to run through the full pipeline


Saving WhatsApp Video 2026-09-04 at 1.58.58 PM.mp4 to WhatsApp Video 2026-09-04 at 1.58.58 PM.mp4
{
  "action_filter_confidence": 0.573,
  "arm_classification": {
    "label": "right",
    "confidence": 1.0
  },
  "pace_classification": {
    "label": "fast",
    "confidence": 1.0
  },
  "closest_pro_match": {
    "player": "Mujeeb Ur Rahman",
    "dtw_distance": 64.193,
    "note": "Lower distance = closer match. Meaningful only if your video's camera angle matches the reference clip's angle."
  },
  "comparison": {
    "movement_quality_similarity_pct": 57.2,
    "symmetry_score": 0.871,
    "coordination_timing_deviation_frames": 0.0,
    "caveats": [
      "All metrics are RELATIVE to the single matched reference clip, not absolute/clinical measurements.",
      "Assumes the uploaded video's camera angle roughly matches the reference dataset's angle."
    ]
  }
}


## 14. Annotated video (skeleton overlay + side report panel)

Purely for demoing to judges that the pipeline is really tracking the bowler's body.


In [27]:
from IPython.display import Video, display

annotated_path, report = generate_annotated_video(
    test_video_path, output_path="/content/annotated_report.mp4")
print(json.dumps(report, indent=2))
if annotated_path:
    display(Video(annotated_path, embed=True, width=760))


{
  "action_filter_confidence": 0.573,
  "arm_classification": {
    "label": "right",
    "confidence": 1.0
  },
  "pace_classification": {
    "label": "fast",
    "confidence": 1.0
  },
  "closest_pro_match": {
    "player": "Mujeeb Ur Rahman",
    "dtw_distance": 64.193,
    "note": "Lower distance = closer match. Meaningful only if your video's camera angle matches the reference clip's angle."
  },
  "comparison": {
    "movement_quality_similarity_pct": 57.2,
    "symmetry_score": 0.871,
    "coordination_timing_deviation_frames": 0.0,
    "caveats": [
      "All metrics are RELATIVE to the single matched reference clip, not absolute/clinical measurements.",
      "Assumes the uploaded video's camera angle roughly matches the reference dataset's angle."
    ]
  }
}


## 15. Export for your backend/frontend + download everything

Generates `config.py`, `pose_utils.py`, `dataset_prep.py`, `action_filter.py`, and
`analyze_video.py` **from the exact code that ran in the cells above** (via
`inspect.getsource`), so there's no separately hand-maintained copy that can drift from
what you actually trained with. Packages those alongside the trained model files into a
zip your frontend's upload endpoint can call `generate_report(video_path)` from directly.


In [28]:
import inspect
import shutil

PACKAGE_DIR = "/content/bowling_style_project_trained"
os.makedirs(PACKAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(PACKAGE_DIR, "models"), exist_ok=True)


def _write_module(filename, header, var_names=(), func_names=()):
    lines = [header.rstrip() + "\n\n"]
    for name in var_names:
        lines.append(f"{name} = {repr(globals()[name])}\n")
    if var_names:
        lines.append("\n")
    for name in func_names:
        lines.append(inspect.getsource(globals()[name]))
        lines.append("\n\n")
    with open(os.path.join(PACKAGE_DIR, filename), "w") as f:
        f.write("".join(lines))


_write_module(
    "config.py",
    '''"""Config generated from the notebook\'s config cell -- paths, feature
settings, and the player -> bowling style label mapping."""
import os

MODELS_DIR = "models"
DATA_DIR = "data"
CACHE_DIR = os.path.join(DATA_DIR, "cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

ACTION_FILTER_MODEL_PATH = os.path.join(MODELS_DIR, "action_filter.keras")
ARM_MODEL_PATH = os.path.join(MODELS_DIR, "arm_classifier.keras")
PACE_MODEL_PATH = os.path.join(MODELS_DIR, "pace_classifier.keras")
ACTION_FILTER_META_PATH = os.path.join(MODELS_DIR, "action_filter_meta.joblib")
ARM_META_PATH = os.path.join(MODELS_DIR, "arm_meta.joblib")
PACE_META_PATH = os.path.join(MODELS_DIR, "pace_meta.joblib")
REFERENCE_LIBRARY_PATH = os.path.join(MODELS_DIR, "reference_library.npz")
FEATURE_CACHE_PATH = os.path.join(CACHE_DIR, "angle_features.npz")
POSE_MODEL_PATH = os.path.join(MODELS_DIR, "pose_landmarker.task")
''',
    var_names=["SEQUENCE_LENGTH", "LM", "ANGLE_NAMES", "ARM_CLASSES",
               "PACE_CLASSES", "ACTION_FILTER_CLASSES", "PLAYER_STYLE"],
)

_write_module(
    "pose_utils.py",
    '''"""MediaPipe Task API pose extraction + hand-crafted joint-angle features.
Generated from the notebook\'s pose-extraction cell."""
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from config import LM, SEQUENCE_LENGTH, ANGLE_NAMES, POSE_MODEL_PATH

_video_options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=mp_vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.4,
    min_tracking_confidence=0.4,
)

POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (24, 26), (25, 27), (26, 28),
    (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32),
]
''',
    func_names=["_angle", "extract_landmark_sequence", "compute_frame_angles",
                "extract_angle_feature_vector", "feature_names",
                "extract_normalized_pose_sequence"],
)

_write_module(
    "dataset_prep.py",
    '''"""Video dataset indexing + grouped CV splits. Only needed if you retrain
locally -- not used at inference time. Generated from the notebook\'s dataset-prep cell."""
import os
from dataclasses import dataclass
from typing import List
from sklearn.model_selection import GroupKFold
from config import VIDEO_DATASET_DIR, PLAYER_STYLE


@dataclass
class ClipEntry:
    path: str
    player: str
    arm: str
    pace: str
''',
    func_names=["index_video_dataset", "grouped_kfold_indices"],
)

_write_module(
    "action_filter.py",
    '''"""Bowling-vs-batting pre-check feature extraction (Task API). Generated from the
notebook\'s action-filter cell. (build_action_filter_model / train_action_filter are
only needed for retraining, not included here -- see the notebook.)"""
import os
import random
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from pose_utils import compute_frame_angles
from config import LM, ANGLE_NAMES, POSE_MODEL_PATH

POSITIVE_PREFIX = "Bowling_action"
NEGATIVE_PREFIX = "batting_stance"

_image_options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.4,
)
''',
    func_names=["extract_image_angle_features", "index_action_dataset"],
)

_write_module(
    "analyze_video.py",
    '''"""Full inference pipeline (Task API). This is what a frontend/backend upload
handler imports: `from analyze_video import load_models, generate_report`.
Call load_models() once at process startup, then generate_report(video_path) per upload.
Generated from the notebook\'s pipeline cell.

Usage (CLI):
    python analyze_video.py --video path/to/clip.mp4
    python analyze_video.py --video path/to/clip.mp4 --annotate --out report.json
"""
import argparse
import json
import os
import joblib
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tensorflow import keras
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

from config import (ARM_MODEL_PATH, PACE_MODEL_PATH, ARM_META_PATH, PACE_META_PATH,
                     ACTION_FILTER_MODEL_PATH, ACTION_FILTER_META_PATH,
                     REFERENCE_LIBRARY_PATH, ARM_CLASSES, PACE_CLASSES, POSE_MODEL_PATH)
from pose_utils import (extract_angle_feature_vector, extract_normalized_pose_sequence,
                         _video_options, POSE_CONNECTIONS)
from action_filter import extract_image_angle_features

action_model = action_scaler = arm_model = arm_scaler = pace_model = pace_scaler = ref = None
''',
    func_names=["load_models", "is_bowling_action", "classify", "find_closest_match",
                "comparison_metrics", "generate_report", "generate_annotated_video"],
)

# CLI entrypoint appended directly (not a notebook-defined function, since it only
# makes sense standalone).
with open(os.path.join(PACKAGE_DIR, "analyze_video.py"), "a") as f:
    f.write('''
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--video", required=True)
    parser.add_argument("--annotate", action="store_true")
    parser.add_argument("--out", default=None)
    args = parser.parse_args()

    load_models()
    if args.annotate:
        annotated_path, report = generate_annotated_video(args.video, "annotated_report.mp4")
        if annotated_path:
            print(f"Annotated video saved: {annotated_path}")
    else:
        report = generate_report(args.video)

    print(json.dumps(report, indent=2))
    if args.out:
        with open(args.out, "w") as fh:
            json.dump(report, fh, indent=2)
        print(f"Report saved: {args.out}")


if __name__ == "__main__":
    main()
''')

with open(os.path.join(PACKAGE_DIR, "requirements.txt"), "w") as f:
    f.write("tensorflow\nmediapipe\nopencv-python\nnumpy\nscikit-learn\n"
            "fastdtw\nscipy\njoblib\n")

with open(os.path.join(PACKAGE_DIR, "README.md"), "w") as f:
    f.write(
        "# Bowling analysis backend\n\n"
        "Generated from the Colab notebook (MediaPipe Task API). Usage:\n\n"
        "```python\nfrom analyze_video import load_models, generate_report\n"
        "load_models()  # once, at startup\nreport = generate_report(video_path)\n```\n\n"
        "Or from the CLI:\n```bash\npip install -r requirements.txt\n"
        "python analyze_video.py --video clip.mp4 --annotate --out report.json\n```\n"
    )

for fname in os.listdir(MODELS_DIR):
    shutil.copy(os.path.join(MODELS_DIR, fname), os.path.join(PACKAGE_DIR, "models", fname))

shutil.copy(POSE_MODEL_PATH, os.path.join(PACKAGE_DIR, "models", "pose_landmarker.task"))

zip_path = "/content/bowling_style_project_trained.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive("/content/bowling_style_project_trained", "zip", PACKAGE_DIR)
print(f"Zipped: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
files.download(zip_path)

Zipped: /content/bowling_style_project_trained.zip (10.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>